In [ ]:
!pip install -q gdown

# Download the dataset from the provided Google Drive link
!gdown --id 1ctTBtolXIWDkZCFJRLChxWtxs0KwcxTs -O /content/dataset.zip

# Unzip the downloaded file directly to /content/
!unzip -q -o /content/dataset.zip -d /content/

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1ctTBtolXIWDkZCFJRLChxWtxs0KwcxTs
From (redirected): https://drive.google.com/uc?id=1ctTBtolXIWDkZCFJRLChxWtxs0KwcxTs&confirm=t&uuid=84f329cf-a61d-4afc-baf3-5464567fafd0
To: /content/dataset.zip
100% 1.84G/1.84G [00:15<00:00, 119MB/s]


In [ ]:
import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torch.optim as optim
from torch.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt

# --- EXACT SAME LOSS FUNCTIONS AS RESUNET++ ---
class SISDRLoss(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps
    def forward(self, preds, targets):
        if preds.dim() == 3: preds = preds.squeeze(1)
        if targets.dim() == 3: targets = targets.squeeze(1)
        preds   = preds - preds.mean(dim=-1, keepdim=True)
        targets = targets - targets.mean(dim=-1, keepdim=True)
        alpha         = (preds * targets).sum(-1, keepdim=True) / (targets.pow(2).sum(-1, keepdim=True) + self.eps)
        target_scaled = alpha * targets
        noise         = preds - target_scaled
        sisdr = 10 * torch.log10(
            (target_scaled.pow(2).sum(-1) + self.eps) /
            (noise.pow(2).sum(-1) + self.eps)
        )
        return -sisdr.mean()

class MRSTFTLoss(nn.Module):
    def __init__(self, fft_sizes=[512, 1024, 2048], hop_sizes=[128, 256, 512], win_lengths=[512, 1024, 2048]):
        super().__init__()
        self.fft_sizes   = fft_sizes
        self.hop_sizes   = hop_sizes
        self.win_lengths = win_lengths
    def forward(self, preds, targets):
        if preds.dim() == 3: preds = preds.squeeze(1)
        if targets.dim() == 3: targets = targets.squeeze(1)
        loss = 0.0
        for n_fft, hop, win in zip(self.fft_sizes, self.hop_sizes, self.win_lengths):
            window = torch.hann_window(win).to(preds.device)
            x_mag  = torch.abs(torch.stft(preds, n_fft=n_fft, hop_length=hop, win_length=win, window=window, return_complex=True, pad_mode='constant'))
            y_mag  = torch.abs(torch.stft(targets, n_fft=n_fft, hop_length=hop, win_length=win, window=window, return_complex=True, pad_mode='constant'))
            sc_loss  = torch.norm(y_mag - x_mag, p='fro') / (torch.norm(y_mag, p='fro') + 1e-7)
            mag_loss = F.l1_loss(torch.log(x_mag + 1e-7), torch.log(y_mag + 1e-7))
            loss += sc_loss + mag_loss
        return loss / len(self.fft_sizes)

class HybridLoss(nn.Module):
    def __init__(self, weight_stft=0.4, weight_sdr=0.6):
        super().__init__()
        self.sisdr_loss  = SISDRLoss()
        self.mrstft_loss = MRSTFTLoss()
        self.weight_stft = weight_stft
        self.weight_sdr  = weight_sdr
    def forward(self, preds, targets):
        loss_sdr  = self.sisdr_loss(preds, targets)
        loss_stft = self.mrstft_loss(preds, targets)
        total     = (self.weight_stft * loss_stft) + (self.weight_sdr * loss_sdr)
        return total, loss_sdr, loss_stft

In [ ]:
class FastOfflineDataset(Dataset):
    def __init__(self, split_dir, target_length=128000):
        self.mix_dir       = os.path.join(split_dir, 'mix')
        self.target_dir    = os.path.join(split_dir, 'target')
        self.target_length = target_length
        all_files          = sorted([f for f in os.listdir(self.mix_dir) if f.endswith('.wav')])
        self.filenames     = [f for f in all_files if os.path.exists(os.path.join(self.target_dir, f))]

    def __len__(self):
        return len(self.filenames)

    def _pad_or_crop(self, w):
        if w.shape[1] > self.target_length:
            return w[:, :self.target_length]
        elif w.shape[1] < self.target_length:
            return F.pad(w, (0, self.target_length - w.shape[1]))
        return w

    def __getitem__(self, idx):
        fname      = self.filenames[idx]
        mix, _     = torchaudio.load(os.path.join(self.mix_dir, fname))
        target, _  = torchaudio.load(os.path.join(self.target_dir, fname))
        if mix.shape[0] > 1: mix = mix.mean(0, keepdim=True)
        if target.shape[0] > 1: target = target.mean(0, keepdim=True)
        return self._pad_or_crop(mix), self._pad_or_crop(target)

def get_dataloaders(dataset_root, batch_size=16, num_workers=2):
    train_ds = FastOfflineDataset(os.path.join(dataset_root, 'train'))
    val_ds   = FastOfflineDataset(os.path.join(dataset_root, 'val'))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader

In [ ]:
class DoubleConv(nn.Module):
    """Standard U-Net Conv Block: Conv -> BN -> ReLU -> Conv -> BN -> ReLU"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.net(x)

class BaselineUNet(nn.Module):
    def __init__(self, n_fft=1024, hop_length=256):
        super().__init__()
        self.n_fft      = n_fft
        self.hop_length = hop_length
        self.window     = nn.Parameter(torch.hann_window(n_fft), requires_grad=False)

        # Encoder (Standard Convs + Pooling)
        self.enc1  = DoubleConv(1, 32) # Input is 1 channel (Magnitude)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2  = DoubleConv(32, 64)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3  = DoubleConv(64, 128)
        self.pool3 = nn.MaxPool2d(2)

        # Bottleneck (Standard Conv, no ASPP)
        self.bottleneck = DoubleConv(128, 256)

        # Decoder (Upsample + Concat + Conv)
        self.up1   = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec1  = DoubleConv(256, 128) # 128 (up) + 128 (skip) = 256

        self.up2   = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2  = DoubleConv(128, 64)  # 64 + 64 = 128

        self.up3   = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec3  = DoubleConv(64, 32)   # 32 + 32 = 64

        # Output Mask
        self.out_mask = nn.Sequential(
            nn.Conv2d(32, 1, kernel_size=1),
            nn.Sigmoid() # Masks are typically 0 to 1 for magnitude
        )

    def forward(self, waveform):
        if waveform.dim() == 3:
            waveform = waveform.squeeze(1)

        # 1. STFT
        stft_out = torch.stft(waveform, n_fft=self.n_fft, hop_length=self.hop_length,
                              window=self.window, return_complex=True, pad_mode='constant')

        # 2. Extract Magnitude and Phase
        mag   = torch.abs(stft_out).unsqueeze(1) # [B, 1, F, T]
        phase = torch.angle(stft_out)            # Save phase for reconstruction

        # Pad to multiple of 8 for 3 pooling layers
        orig_F, orig_T = mag.shape[2], mag.shape[3]
        pad_F = (8 - (orig_F % 8)) % 8
        pad_T = (8 - (orig_T % 8)) % 8
        x = F.pad(mag, (0, pad_T, 0, pad_F))

        # 3. Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))

        # 4. Bottleneck
        b = self.bottleneck(self.pool3(e3))

        # 5. Decoder with Skip Connections
        d1 = self.dec1(torch.cat([self.up1(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d1), e2], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d2), e1], dim=1))

        # 6. Predict Mask and Apply to Magnitude
        mask = self.out_mask(d3)
        mask = mask[:, :, :orig_F, :orig_T] # Remove padding
        mag_orig = mag[:, :, :orig_F, :orig_T]

        clean_mag = mag_orig * mask
        clean_mag = clean_mag.squeeze(1) # Back to [B, F, T]

        # 7. ISTFT (Combine Clean Magnitude with NOISY Mix Phase)
        out_complex = torch.polar(clean_mag, phase)
        return torch.istft(out_complex, n_fft=self.n_fft, hop_length=self.hop_length,
                           window=self.window, length=waveform.shape[1]).unsqueeze(1)

In [ ]:
import os

print("🔍 Searching for your dataset folders...\n")

found_root = None
for root, dirs, files in os.walk('/content'):
    # If we find a folder that contains BOTH 'train' and 'val'
    if 'train' in dirs and 'val' in dirs:
        found_root = root
        break

if found_root:
    print(f"✅ Found it! The exact path to your dataset is:")
    print(f"   {found_root}\n")
    print(f"👉 Copy that path and paste it into DATASET_ROOT = '{found_root}'")
else:
    print("❌ Could not find a folder containing 'train' and 'val'.")
    print("Let's see what is actually inside /content/synthetic_dataset_v2:")
    !ls -la /content/synthetic_dataset_v2

🔍 Searching for your dataset folders...

✅ Found it! The exact path to your dataset is:
   /content/content/synthetic_dataset_v2

👉 Copy that path and paste it into DATASET_ROOT = '/content/content/synthetic_dataset_v2'


In [ ]:
def train_baseline_unet():
    # 0 A100 SPEED UP: Enable CUDNN Benchmark and TF32
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"0 Launching Baseline U-Net on: {device}")

    # --- SAVE DIRECTORIES ---
    BASE_DIR       = '/content/drive/MyDrive'
    CHECKPOINT_DIR = os.path.join(BASE_DIR, 'Baseline_UNet_Results')
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

    # ⚠️ UPDATE THIS IF NEEDED
    DATASET_ROOT = "/content/content/synthetic_dataset_v2"

    # 0 A100 SPEED UP: Increased batch_size to 32 and num_workers to 8
    train_loader, val_loader = get_dataloaders(DATASET_ROOT, batch_size=32, num_workers=8)

    model     = BaselineUNet().to(device)
    criterion = HybridLoss(weight_stft=0.4, weight_sdr=0.6).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
    epochs    = 50
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler    = GradScaler('cuda')

    best_val_sdr    = float('-inf')
    checkpoint_path = os.path.join(CHECKPOINT_DIR, 'best_baseline_unet.pth')
    history_path    = os.path.join(CHECKPOINT_DIR, 'unet_history.json')
    history         = {'train_loss': [], 'val_loss': [], 'train_sdr': [], 'val_sdr': []}

    for epoch in range(1, epochs + 1):
        # --- Train ---
        model.train()
        train_loss, train_sdr = 0.0, 0.0
        loop = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs} [Train]", leave=False)
        for mixed, target in loop:
            mixed, target = mixed.to(device), target.to(device)
            optimizer.zero_grad()
            with autocast('cuda'):
                preds = model(mixed)
                loss, loss_sdr, _ = criterion(preds, target)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()
            train_sdr  += -loss_sdr.item()
            loop.set_postfix(loss=f"{loss.item():.4f}", sdr=f"{-loss_sdr.item():.3f}")

        avg_train_loss = train_loss / len(train_loader)
        avg_train_sdr  = train_sdr  / len(train_loader)

        # --- Validate ---
        model.eval()
        val_loss, val_sdr = 0.0, 0.0
        with torch.no_grad():
            for mixed, target in tqdm(val_loader, desc=f"Epoch {epoch}/{epochs} [Val]", leave=False):
                mixed, target = mixed.to(device), target.to(device)
                with autocast('cuda'):
                    preds = model(mixed)
                    loss, loss_sdr, _ = criterion(preds, target)
                val_loss += loss.item()
                val_sdr  += -loss_sdr.item()

        avg_val_loss = val_loss / len(val_loader)
        avg_val_sdr  = val_sdr  / len(val_loader)

        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['train_sdr'].append(avg_train_sdr)
        history['val_sdr'].append(avg_val_sdr)

        # Save history JSON
        with open(history_path, 'w') as f: json.dump(history, f)
        scheduler.step()

        print(f"Epoch [{epoch}/{epochs}] | Train SDR: {avg_train_sdr:.3f} dB | Val SDR: {avg_val_sdr:.3f} dB")

        # Save Best Model
        if avg_val_sdr > best_val_sdr:
            best_val_sdr = avg_val_sdr
            torch.save(model.state_dict(), checkpoint_path)
            print(f"   ⭐ Best Baseline U-Net saved | Val SI-SDR: {best_val_sdr:.3f} dB")

    # ==========================================
    # 🎉 TRAINING COMPLETE - GENERATE PLOTS
    # ==========================================
    print("Training Complete! Generating plots...")

    # 1. Loss Plot
    plt.figure(figsize=(10, 5))
    plt.plot(history['train_loss'], label='Train Loss', color='blue')
    plt.plot(history['val_loss'], label='Val Loss', color='orange')
    plt.title('Baseline U-Net: Hybrid Loss Over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(CHECKPOINT_DIR, 'loss_plot.png'))
    plt.close()

    # 2. SI-SDR Plot
    plt.figure(figsize=(10, 5))
    plt.plot(history['train_sdr'], label='Train SI-SDR', color='green')
    plt.plot(history['val_sdr'], label='Val SI-SDR', color='red')
    plt.title('Baseline U-Net: SI-SDR Improvement Over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('SI-SDR (dB)')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(CHECKPOINT_DIR, 'sdr_plot.png'))
    plt.close()

    print(f"✅ All files (Model, JSON History, and Plots) securely saved to {CHECKPOINT_DIR}")

# 0 RUN IT
train_baseline_unet()

0 Launching Baseline U-Net on: cuda


Epoch [1/50] | Train SDR: 1.408 dB | Val SDR: 1.777 dB
   ⭐ Best Baseline U-Net saved | Val SI-SDR: 1.777 dB


Epoch [2/50] | Train SDR: 2.723 dB | Val SDR: 2.160 dB
   ⭐ Best Baseline U-Net saved | Val SI-SDR: 2.160 dB


Epoch [3/50] | Train SDR: 3.221 dB | Val SDR: 1.898 dB


Epoch [4/50] | Train SDR: 3.537 dB | Val SDR: 2.799 dB
   ⭐ Best Baseline U-Net saved | Val SI-SDR: 2.799 dB


...

Epoch [46/50] | Train SDR: 7.581 dB | Val SDR: 3.188 dB


Epoch [47/50] | Train SDR: 7.602 dB | Val SDR: 3.264 dB


Epoch [48/50] | Train SDR: 7.609 dB | Val SDR: 3.227 dB


Epoch [49/50] | Train SDR: 7.615 dB | Val SDR: 3.221 dB


Epoch [50/50] | Train SDR: 7.625 dB | Val SDR: 3.233 dB
Training Complete! Generating plots...
✅ All files (Model, JSON History, and Plots) securely saved to /content/drive/MyDrive/Baseline_UNet_Results


convtsaent

In [ ]:
import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torch.optim as optim
from torch.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt

# --- EXACT SAME LOSS FUNCTIONS ---
class SISDRLoss(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps
    def forward(self, preds, targets):
        if preds.dim() == 3: preds = preds.squeeze(1)
        if targets.dim() == 3: targets = targets.squeeze(1)
        preds   = preds - preds.mean(dim=-1, keepdim=True)
        targets = targets - targets.mean(dim=-1, keepdim=True)
        alpha         = (preds * targets).sum(-1, keepdim=True) / (targets.pow(2).sum(-1, keepdim=True) + self.eps)
        target_scaled = alpha * targets
        noise         = preds - target_scaled
        sisdr = 10 * torch.log10(
            (target_scaled.pow(2).sum(-1) + self.eps) /
            (noise.pow(2).sum(-1) + self.eps)
        )
        return -sisdr.mean()

class MRSTFTLoss(nn.Module):
    def __init__(self, fft_sizes=[512, 1024, 2048], hop_sizes=[128, 256, 512], win_lengths=[512, 1024, 2048]):
        super().__init__()
        self.fft_sizes   = fft_sizes
        self.hop_sizes   = hop_sizes
        self.win_lengths = win_lengths
    def forward(self, preds, targets):
        if preds.dim() == 3: preds = preds.squeeze(1)
        if targets.dim() == 3: targets = targets.squeeze(1)
        loss = 0.0
        for n_fft, hop, win in zip(self.fft_sizes, self.hop_sizes, self.win_lengths):
            window = torch.hann_window(win).to(preds.device)
            x_mag  = torch.abs(torch.stft(preds, n_fft=n_fft, hop_length=hop, win_length=win, window=window, return_complex=True, pad_mode='constant'))
            y_mag  = torch.abs(torch.stft(targets, n_fft=n_fft, hop_length=hop, win_length=win, window=window, return_complex=True, pad_mode='constant'))
            sc_loss  = torch.norm(y_mag - x_mag, p='fro') / (torch.norm(y_mag, p='fro') + 1e-7)
            mag_loss = F.l1_loss(torch.log(x_mag + 1e-7), torch.log(y_mag + 1e-7))
            loss += sc_loss + mag_loss
        return loss / len(self.fft_sizes)

class HybridLoss(nn.Module):
    def __init__(self, weight_stft=0.4, weight_sdr=0.6):
        super().__init__()
        self.sisdr_loss  = SISDRLoss()
        self.mrstft_loss = MRSTFTLoss()
        self.weight_stft = weight_stft
        self.weight_sdr  = weight_sdr
    def forward(self, preds, targets):
        loss_sdr  = self.sisdr_loss(preds, targets)
        loss_stft = self.mrstft_loss(preds, targets)
        total     = (self.weight_stft * loss_stft) + (self.weight_sdr * loss_sdr)
        return total, loss_sdr, loss_stft

In [ ]:
class FastOfflineDataset(Dataset):
    def __init__(self, split_dir, target_length=128000):
        self.mix_dir       = os.path.join(split_dir, 'mix')
        self.target_dir    = os.path.join(split_dir, 'target')
        self.target_length = target_length
        all_files          = sorted([f for f in os.listdir(self.mix_dir) if f.endswith('.wav')])
        self.filenames     = [f for f in all_files if os.path.exists(os.path.join(self.target_dir, f))]

    def __len__(self):
        return len(self.filenames)

    def _pad_or_crop(self, w):
        if w.shape[1] > self.target_length:
            return w[:, :self.target_length]
        elif w.shape[1] < self.target_length:
            return F.pad(w, (0, self.target_length - w.shape[1]))
        return w

    def __getitem__(self, idx):
        fname      = self.filenames[idx]
        mix, _     = torchaudio.load(os.path.join(self.mix_dir, fname))
        target, _  = torchaudio.load(os.path.join(self.target_dir, fname))
        if mix.shape[0] > 1: mix = mix.mean(0, keepdim=True)
        if target.shape[0] > 1: target = target.mean(0, keepdim=True)
        return self._pad_or_crop(mix), self._pad_or_crop(target)

def get_dataloaders(dataset_root, batch_size=16, num_workers=2):
    train_ds = FastOfflineDataset(os.path.join(dataset_root, 'train'))
    val_ds   = FastOfflineDataset(os.path.join(dataset_root, 'val'))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader

In [ ]:
from torchaudio.models import ConvTasNet

class BaselineConvTasNet(nn.Module):
    def __init__(self):
        super().__init__()
        # Using the official PyTorch torchaudio parameter names!
        self.model = ConvTasNet(
            num_sources=1,             # We are extracting 1 target source
            enc_kernel_size=16,        # Length of the filters
            enc_num_feats=256,         # Number of filters in encoder
            msk_kernel_size=3,         # Kernel size of TCN blocks
            msk_num_feats=128,         # Number of channels in bottleneck
            msk_num_hidden_feats=256,  # Number of channels in TCN blocks
            msk_num_layers=8,          # Number of dilations per repeat
            msk_num_stacks=2,          # Number of times to repeat the TCN block structure
            msk_activate='sigmoid'     # Mask activation function
        )

    def forward(self, waveform):
        # Conv-TasNet requires [Batch, Channels, Time]
        if waveform.dim() == 2:
            waveform = waveform.unsqueeze(1)

        # The torchaudio implementation outputs [Batch, num_sources, Time]
        out = self.model(waveform)

        return out

In [ ]:
def train_conv_tasnet():
    # 🚀 A100 SPEED UP: Enable CUDNN Benchmark and TF32
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"🚀 Launching Baseline Conv-TasNet on: {device}")

    # --- SAVE DIRECTORIES ---
    BASE_DIR       = '/content/drive/MyDrive'
    CHECKPOINT_DIR = os.path.join(BASE_DIR, 'ConvTasNet_Results')
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

    # ⚠️ UPDATE THIS IF NEEDED
    DATASET_ROOT = "/content/content/synthetic_dataset_v2"
    # 🚀 A100 SPEED UP: Increased batch_size to 16 and num_workers to 8
    train_loader, val_loader = get_dataloaders(DATASET_ROOT, batch_size=16, num_workers=8)

    model     = BaselineConvTasNet().to(device)
    criterion = HybridLoss(weight_stft=0.4, weight_sdr=0.6).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
    epochs    = 50
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler    = GradScaler('cuda')

    best_val_sdr    = float('-inf')
    checkpoint_path = os.path.join(CHECKPOINT_DIR, 'best_convtasnet.pth')
    history_path    = os.path.join(CHECKPOINT_DIR, 'convtasnet_history.json')
    history         = {'train_loss': [], 'val_loss': [], 'train_sdr': [], 'val_sdr': []}

    # Early stopping parameters
    patience = 10
    no_improve_epochs = 0

    for epoch in range(1, epochs + 1):
        # --- Train ---
        model.train()
        train_loss, train_sdr = 0.0, 0.0
        loop = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs} [Train]", leave=False)
        for mixed, target in loop:
            mixed, target = mixed.to(device), target.to(device)
            optimizer.zero_grad()
            with autocast('cuda'):
                preds = model(mixed)
                loss, loss_sdr, _ = criterion(preds, target)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()
            train_sdr  += -loss_sdr.item()
            loop.set_postfix(loss=f"{loss.item():.4f}", sdr=f"{-loss_sdr.item():.3f}")

        avg_train_loss = train_loss / len(train_loader)
        avg_train_sdr  = train_sdr  / len(train_loader)

        # --- Validate ---
        model.eval()
        val_loss, val_sdr = 0.0, 0.0
        with torch.no_grad():
            for mixed, target in tqdm(val_loader, desc=f"Epoch {epoch}/{epochs} [Val]", leave=False):
                mixed, target = mixed.to(device), target.to(device)
                with autocast('cuda'):
                    preds = model(mixed)
                    loss, loss_sdr, _ = criterion(preds, target)
                val_loss += loss.item()
                val_sdr  += -loss_sdr.item()

        avg_val_loss = val_loss / len(val_loader)
        avg_val_sdr  = val_sdr  / len(val_loader)

        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['train_sdr'].append(avg_train_sdr)
        history['val_sdr'].append(avg_val_sdr)

        # Save history JSON
        with open(history_path, 'w') as f: json.dump(history, f)
        scheduler.step()

        print(f"Epoch [{epoch}/{epochs}] | Train SDR: {avg_train_sdr:.3f} dB | Val SDR: {avg_val_sdr:.3f} dB")

        # Save Best Model
        if avg_val_sdr > best_val_sdr:
            best_val_sdr = avg_val_sdr
            torch.save(model.state_dict(), checkpoint_path)
            print(f"   🌟 Best Conv-TasNet saved | Val SI-SDR: {best_val_sdr:.3f} dB")
            no_improve_epochs = 0
        else:
            no_improve_epochs += 1
            print(f"   ⚠️ No improvement for {no_improve_epochs} epoch(s).")
            if no_improve_epochs >= patience:
                print(f"🛑 Early stopping triggered! No improvement for {patience} epochs.")
                break

    # ==========================================
    # 🎉 TRAINING COMPLETE - GENERATE PLOTS
    # ==========================================
    print("Training Complete! Generating plots...")

    # 1. Loss Plot
    plt.figure(figsize=(10, 5))
    plt.plot(history['train_loss'], label='Train Loss', color='blue')
    plt.plot(history['val_loss'], label='Val Loss', color='orange')
    plt.title('Conv-TasNet: Hybrid Loss Over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(CHECKPOINT_DIR, 'loss_plot.png'))
    plt.close()

    # 2. SI-SDR Plot
    plt.figure(figsize=(10, 5))
    plt.plot(history['train_sdr'], label='Train SI-SDR', color='green')
    plt.plot(history['val_sdr'], label='Val SI-SDR', color='red')
    plt.title('Conv-TasNet: SI-SDR Improvement Over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('SI-SDR (dB)')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(CHECKPOINT_DIR, 'sdr_plot.png'))
    plt.close()

    print(f"✅ All files (Model, JSON History, and Plots) securely saved to {CHECKPOINT_DIR}")

# 🚀 RUN IT
train_conv_tasnet()

🚀 Launching Baseline Conv-TasNet on: cuda


Epoch [1/50] | Train SDR: -0.081 dB | Val SDR: 1.303 dB
   🌟 Best Conv-TasNet saved | Val SI-SDR: 1.303 dB


Epoch [2/50] | Train SDR: 1.225 dB | Val SDR: 1.399 dB
   🌟 Best Conv-TasNet saved | Val SI-SDR: 1.399 dB


Epoch [3/50] | Train SDR: 1.422 dB | Val SDR: 1.718 dB
   🌟 Best Conv-TasNet saved | Val SI-SDR: 1.718 dB


Epoch [4/50] | Train SDR: 1.556 dB | Val SDR: 1.875 dB
   🌟 Best Conv-TasNet saved | Val SI-SDR: 1.875 dB


...

Epoch [43/50] | Train SDR: 4.433 dB | Val SDR: 3.304 dB
   ⚠️ No improvement for 6 epoch(s).


Epoch [44/50] | Train SDR: 4.437 dB | Val SDR: 3.288 dB
   ⚠️ No improvement for 7 epoch(s).


Epoch [45/50] | Train SDR: 4.461 dB | Val SDR: 3.292 dB
   ⚠️ No improvement for 8 epoch(s).


Epoch [46/50] | Train SDR: 4.468 dB | Val SDR: 3.294 dB
   ⚠️ No improvement for 9 epoch(s).


Epoch [47/50] | Train SDR: 4.479 dB | Val SDR: 3.294 dB
   ⚠️ No improvement for 10 epoch(s).
🛑 Early stopping triggered! No improvement for 10 epochs.
Training Complete! Generating plots...
✅ All files (Model, JSON History, and Plots) securely saved to /content/drive/MyDrive/ConvTasNet_Results


demus

In [ ]:
import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torch.optim as optim
from torch.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt

# --- EXACT SAME LOSS FUNCTIONS ---
class SISDRLoss(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps
    def forward(self, preds, targets):
        if preds.dim() == 3: preds = preds.squeeze(1)
        if targets.dim() == 3: targets = targets.squeeze(1)
        preds   = preds - preds.mean(dim=-1, keepdim=True)
        targets = targets - targets.mean(dim=-1, keepdim=True)
        alpha         = (preds * targets).sum(-1, keepdim=True) / (targets.pow(2).sum(-1, keepdim=True) + self.eps)
        target_scaled = alpha * targets
        noise         = preds - target_scaled
        sisdr = 10 * torch.log10(
            (target_scaled.pow(2).sum(-1) + self.eps) /
            (noise.pow(2).sum(-1) + self.eps)
        )
        return -sisdr.mean()

class MRSTFTLoss(nn.Module):
    def __init__(self, fft_sizes=[512, 1024, 2048], hop_sizes=[128, 256, 512], win_lengths=[512, 1024, 2048]):
        super().__init__()
        self.fft_sizes   = fft_sizes
        self.hop_sizes   = hop_sizes
        self.win_lengths = win_lengths
    def forward(self, preds, targets):
        if preds.dim() == 3: preds = preds.squeeze(1)
        if targets.dim() == 3: targets = targets.squeeze(1)
        loss = 0.0
        for n_fft, hop, win in zip(self.fft_sizes, self.hop_sizes, self.win_lengths):
            window = torch.hann_window(win).to(preds.device)
            x_mag  = torch.abs(torch.stft(preds, n_fft=n_fft, hop_length=hop, win_length=win, window=window, return_complex=True, pad_mode='constant'))
            y_mag  = torch.abs(torch.stft(targets, n_fft=n_fft, hop_length=hop, win_length=win, window=window, return_complex=True, pad_mode='constant'))
            sc_loss  = torch.norm(y_mag - x_mag, p='fro') / (torch.norm(y_mag, p='fro') + 1e-7)
            mag_loss = F.l1_loss(torch.log(x_mag + 1e-7), torch.log(y_mag + 1e-7))
            loss += sc_loss + mag_loss
        return loss / len(self.fft_sizes)

class HybridLoss(nn.Module):
    def __init__(self, weight_stft=0.4, weight_sdr=0.6):
        super().__init__()
        self.sisdr_loss  = SISDRLoss()
        self.mrstft_loss = MRSTFTLoss()
        self.weight_stft = weight_stft
        self.weight_sdr  = weight_sdr
    def forward(self, preds, targets):
        loss_sdr  = self.sisdr_loss(preds, targets)
        loss_stft = self.mrstft_loss(preds, targets)
        total     = (self.weight_stft * loss_stft) + (self.weight_sdr * loss_sdr)
        return total, loss_sdr, loss_stft

In [ ]:
class FastOfflineDataset(Dataset):
    def __init__(self, split_dir, target_length=128000):
        self.mix_dir       = os.path.join(split_dir, 'mix')
        self.target_dir    = os.path.join(split_dir, 'target')
        self.target_length = target_length
        all_files          = sorted([f for f in os.listdir(self.mix_dir) if f.endswith('.wav')])
        self.filenames     = [f for f in all_files if os.path.exists(os.path.join(self.target_dir, f))]

    def __len__(self):
        return len(self.filenames)

    def _pad_or_crop(self, w):
        if w.shape[1] > self.target_length:
            return w[:, :self.target_length]
        elif w.shape[1] < self.target_length:
            return F.pad(w, (0, self.target_length - w.shape[1]))
        return w

    def __getitem__(self, idx):
        fname      = self.filenames[idx]
        mix, _     = torchaudio.load(os.path.join(self.mix_dir, fname))
        target, _  = torchaudio.load(os.path.join(self.target_dir, fname))
        if mix.shape[0] > 1: mix = mix.mean(0, keepdim=True)
        if target.shape[0] > 1: target = target.mean(0, keepdim=True)
        return self._pad_or_crop(mix), self._pad_or_crop(target)

def get_dataloaders(dataset_root, batch_size=16, num_workers=2):
    train_ds = FastOfflineDataset(os.path.join(dataset_root, 'train'))
    val_ds   = FastOfflineDataset(os.path.join(dataset_root, 'val'))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader

In [ ]:
def train_demucs():
    # 🚀 A100 SPEED UP: Enable CUDNN Benchmark and TF32
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"🚀 Launching Baseline Demucs on: {device}")

    # --- SAVE DIRECTORIES ---
    BASE_DIR       = '/content/drive/MyDrive'
    CHECKPOINT_DIR = os.path.join(BASE_DIR, 'Demucs_Results')
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

    # ⚠️ UPDATE THIS IF NEEDED
    DATASET_ROOT = "/content/content/synthetic_dataset_v2"
    # 🚀 A100 SPEED UP: Increased batch_size to 16 and num_workers to 8
    train_loader, val_loader = get_dataloaders(DATASET_ROOT, batch_size=16, num_workers=8)

    model     = BaselineDemucs().to(device)
    criterion = HybridLoss(weight_stft=0.4, weight_sdr=0.6).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
    epochs    = 50
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler    = GradScaler('cuda')

    best_val_sdr    = float('-inf')
    checkpoint_path = os.path.join(CHECKPOINT_DIR, 'best_demucs.pth')
    history_path    = os.path.join(CHECKPOINT_DIR, 'demucs_history.json')
    history         = {'train_loss': [], 'val_loss': [], 'train_sdr': [], 'val_sdr': []}

    # Early stopping parameters
    patience = 10
    no_improve_epochs = 0

    for epoch in range(1, epochs + 1):
        # --- Train ---
        model.train()
        train_loss, train_sdr = 0.0, 0.0
        loop = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs} [Train]", leave=False)
        for mixed, target in loop:
            mixed, target = mixed.to(device), target.to(device)
            optimizer.zero_grad()
            with autocast('cuda'):
                preds = model(mixed)
                loss, loss_sdr, _ = criterion(preds, target)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()
            train_sdr  += -loss_sdr.item()
            loop.set_postfix(loss=f"{loss.item():.4f}", sdr=f"{-loss_sdr.item():.3f}")

        avg_train_loss = train_loss / len(train_loader)
        avg_train_sdr  = train_sdr  / len(train_loader)

        # --- Validate ---
        model.eval()
        val_loss, val_sdr = 0.0, 0.0
        with torch.no_grad():
            for mixed, target in tqdm(val_loader, desc=f"Epoch {epoch}/{epochs} [Val]", leave=False):
                mixed, target = mixed.to(device), target.to(device)
                with autocast('cuda'):
                    preds = model(mixed)
                    loss, loss_sdr, _ = criterion(preds, target)
                val_loss += loss.item()
                val_sdr  += -loss_sdr.item()

        avg_val_loss = val_loss / len(val_loader)
        avg_val_sdr  = val_sdr  / len(val_loader)

        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['train_sdr'].append(avg_train_sdr)
        history['val_sdr'].append(avg_val_sdr)

        # Save history JSON
        with open(history_path, 'w') as f: json.dump(history, f)
        scheduler.step()

        print(f"Epoch [{epoch}/{epochs}] | Train SDR: {avg_train_sdr:.3f} dB | Val SDR: {avg_val_sdr:.3f} dB")

        # Save Best Model
        if avg_val_sdr > best_val_sdr:
            best_val_sdr = avg_val_sdr
            torch.save(model.state_dict(), checkpoint_path)
            print(f"   🌟 Best Demucs saved | Val SI-SDR: {best_val_sdr:.3f} dB")
            no_improve_epochs = 0
        else:
            no_improve_epochs += 1
            print(f"   ⚠️ No improvement for {no_improve_epochs} epoch(s).")
            if no_improve_epochs >= patience:
                print(f"🛑 Early stopping triggered! No improvement for {patience} epochs.")
                break

    # ==========================================
    # 🎉 TRAINING COMPLETE - GENERATE PLOTS
    # ==========================================
    print("Training Complete! Generating plots...")

    # 1. Loss Plot
    plt.figure(figsize=(10, 5))
    plt.plot(history['train_loss'], label='Train Loss', color='blue')
    plt.plot(history['val_loss'], label='Val Loss', color='orange')
    plt.title('Baseline Demucs: Hybrid Loss Over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(CHECKPOINT_DIR, 'loss_plot.png'))
    plt.close()

    # 2. SI-SDR Plot
    plt.figure(figsize=(10, 5))
    plt.plot(history['train_sdr'], label='Train SI-SDR', color='green')
    plt.plot(history['val_sdr'], label='Val SI-SDR', color='red')
    plt.title('Baseline Demucs: SI-SDR Improvement Over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('SI-SDR (dB)')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(CHECKPOINT_DIR, 'sdr_plot.png'))
    plt.close()

    print(f"✅ All files (Model, JSON History, and Plots) securely saved to {CHECKPOINT_DIR}")

# 🚀 RUN IT
train_demucs()

🚀 Launching Baseline Demucs on: cuda


Epoch [1/50] | Train SDR: -2.709 dB | Val SDR: 0.989 dB
   🌟 Best Demucs saved | Val SI-SDR: 0.989 dB


Epoch [2/50] | Train SDR: 0.408 dB | Val SDR: 1.102 dB
   🌟 Best Demucs saved | Val SI-SDR: 1.102 dB


Epoch [3/50] | Train SDR: 0.550 dB | Val SDR: 1.236 dB
   🌟 Best Demucs saved | Val SI-SDR: 1.236 dB


Epoch [4/50] | Train SDR: 0.801 dB | Val SDR: 1.605 dB
   🌟 Best Demucs saved | Val SI-SDR: 1.605 dB


...

Epoch [37/50] | Train SDR: 4.735 dB | Val SDR: 2.301 dB
   ⚠️ No improvement for 6 epoch(s).


Epoch [38/50] | Train SDR: 4.774 dB | Val SDR: 2.301 dB
   ⚠️ No improvement for 7 epoch(s).


Epoch [39/50] | Train SDR: 4.824 dB | Val SDR: 2.274 dB
   ⚠️ No improvement for 8 epoch(s).


Epoch [40/50] | Train SDR: 4.853 dB | Val SDR: 2.269 dB
   ⚠️ No improvement for 9 epoch(s).


Epoch [41/50] | Train SDR: 4.889 dB | Val SDR: 2.268 dB
   ⚠️ No improvement for 10 epoch(s).
🛑 Early stopping triggered! No improvement for 10 epochs.
Training Complete! Generating plots...
✅ All files (Model, JSON History, and Plots) securely saved to /content/drive/MyDrive/Demucs_Results


reunet++


In [ ]:
import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torch.optim as optim
from torch.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt

# ==========================================
# 1. IDENTICAL LOSS FUNCTIONS
# ==========================================
class SISDRLoss(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps
    def forward(self, preds, targets):
        if preds.dim() == 3: preds = preds.squeeze(1)
        if targets.dim() == 3: targets = targets.squeeze(1)
        preds   = preds - preds.mean(dim=-1, keepdim=True)
        targets = targets - targets.mean(dim=-1, keepdim=True)
        alpha         = (preds * targets).sum(-1, keepdim=True) / (targets.pow(2).sum(-1, keepdim=True) + self.eps)
        target_scaled = alpha * targets
        noise         = preds - target_scaled
        sisdr = 10 * torch.log10(
            (target_scaled.pow(2).sum(-1) + self.eps) /
            (noise.pow(2).sum(-1) + self.eps)
        )
        return -sisdr.mean()

class MRSTFTLoss(nn.Module):
    def __init__(self, fft_sizes=[512, 1024, 2048], hop_sizes=[128, 256, 512], win_lengths=[512, 1024, 2048]):
        super().__init__()
        self.fft_sizes   = fft_sizes
        self.hop_sizes   = hop_sizes
        self.win_lengths = win_lengths
    def forward(self, preds, targets):
        if preds.dim() == 3: preds = preds.squeeze(1)
        if targets.dim() == 3: targets = targets.squeeze(1)
        loss = 0.0
        for n_fft, hop, win in zip(self.fft_sizes, self.hop_sizes, self.win_lengths):
            window = torch.hann_window(win).to(preds.device)
            x_mag  = torch.abs(torch.stft(preds, n_fft=n_fft, hop_length=hop, win_length=win, window=window, return_complex=True, pad_mode='constant'))
            y_mag  = torch.abs(torch.stft(targets, n_fft=n_fft, hop_length=hop, win_length=win, window=window, return_complex=True, pad_mode='constant'))
            sc_loss  = torch.norm(y_mag - x_mag, p='fro') / (torch.norm(y_mag, p='fro') + 1e-7)
            mag_loss = F.l1_loss(torch.log(x_mag + 1e-7), torch.log(y_mag + 1e-7))
            loss += sc_loss + mag_loss
        return loss / len(self.fft_sizes)

class HybridLoss(nn.Module):
    def __init__(self, weight_stft=0.4, weight_sdr=0.6):
        super().__init__()
        self.sisdr_loss  = SISDRLoss()
        self.mrstft_loss = MRSTFTLoss()
        self.weight_stft = weight_stft
        self.weight_sdr  = weight_sdr
    def forward(self, preds, targets):
        loss_sdr  = self.sisdr_loss(preds, targets)
        loss_stft = self.mrstft_loss(preds, targets)
        total     = (self.weight_stft * loss_stft) + (self.weight_sdr * loss_sdr)
        return total, loss_sdr, loss_stft

# ==========================================
# 2. IDENTICAL DATALOADER (BATCH SIZE CONTROL)
# ==========================================
class FastOfflineDataset(Dataset):
    def __init__(self, split_dir, target_length=128000):
        self.mix_dir       = os.path.join(split_dir, 'mix')
        self.target_dir    = os.path.join(split_dir, 'target')
        self.target_length = target_length
        all_files          = sorted([f for f in os.listdir(self.mix_dir) if f.endswith('.wav')])
        self.filenames     = [f for f in all_files if os.path.exists(os.path.join(self.target_dir, f))]

    def __len__(self):
        return len(self.filenames)

    def _pad_or_crop(self, w):
        if w.shape[1] > self.target_length:
            return w[:, :self.target_length]
        elif w.shape[1] < self.target_length:
            return F.pad(w, (0, self.target_length - w.shape[1]))
        return w

    def __getitem__(self, idx):
        fname      = self.filenames[idx]
        mix, _     = torchaudio.load(os.path.join(self.mix_dir, fname))
        target, _  = torchaudio.load(os.path.join(self.target_dir, fname))
        if mix.shape[0] > 1: mix = mix.mean(0, keepdim=True)
        if target.shape[0] > 1: target = target.mean(0, keepdim=True)
        return self._pad_or_crop(mix), self._pad_or_crop(target)

def get_dataloaders(dataset_root, batch_size=4, num_workers=2): # BATCH SIZE 4 SET HERE
    train_ds = FastOfflineDataset(os.path.join(dataset_root, 'train'))
    val_ds   = FastOfflineDataset(os.path.join(dataset_root, 'val'))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader

In [ ]:
# ==========================================
# 3. YOUR PROPOSED ARCHITECTURE
# ==========================================
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1    = nn.Conv2d(in_channels,  out_channels, 3, padding=1)
        self.bn1      = nn.BatchNorm2d(out_channels)
        self.conv2    = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.bn2      = nn.BatchNorm2d(out_channels)
        self.se       = SEBlock(out_channels)
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1),
                nn.BatchNorm2d(out_channels)
            )
    def forward(self, x):
        residual = self.shortcut(x)
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        out += residual
        return F.relu(out)

class ASPP(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1    = nn.Conv2d(in_channels, out_channels, 1)
        self.conv2    = nn.Conv2d(in_channels, out_channels, 3, padding=6,  dilation=6)
        self.conv3    = nn.Conv2d(in_channels, out_channels, 3, padding=12, dilation=12)
        self.conv4    = nn.Conv2d(in_channels, out_channels, 3, padding=18, dilation=18)
        self.out_conv = nn.Conv2d(out_channels * 4, out_channels, 1)
    def forward(self, x):
        x1 = F.relu(self.conv1(x))
        x2 = F.relu(self.conv2(x))
        x3 = F.relu(self.conv3(x))
        x4 = F.relu(self.conv4(x))
        return self.out_conv(torch.cat([x1, x2, x3, x4], dim=1))

class ResUNetPlusPlus(nn.Module):
    def __init__(self, n_fft=1024, hop_length=256):
        super().__init__()
        self.n_fft      = n_fft
        self.hop_length = hop_length
        self.window     = nn.Parameter(torch.hann_window(n_fft), requires_grad=False)

        # Encoder
        self.enc1  = ResidualBlock(2,   32)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2  = ResidualBlock(32,  64)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3  = ResidualBlock(64,  128)
        self.pool3 = nn.MaxPool2d(2)

        # ASPP Bottleneck
        self.aspp  = ASPP(128, 256)

        # Decoder
        self.up1   = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec1  = ResidualBlock(256, 128)
        self.up2   = nn.ConvTranspose2d(128,  64, 2, stride=2)
        self.dec2  = ResidualBlock(128,  64)
        self.up3   = nn.ConvTranspose2d(64,   32, 2, stride=2)
        self.dec3  = ResidualBlock(64,   32)

        self.out_mask = nn.Conv2d(32, 2, 1)

    def forward(self, waveform):
        if waveform.dim() == 3:
            waveform = waveform.squeeze(1)

        # STFT to Complex Spectrogram
        stft_out   = torch.stft(waveform, n_fft=self.n_fft, hop_length=self.hop_length,
                                window=self.window, return_complex=True, pad_mode='constant')
        real_part  = stft_out.real.unsqueeze(1)
        imag_part  = stft_out.imag.unsqueeze(1)
        x          = torch.cat([real_part, imag_part], dim=1)

        orig_F, orig_T = x.shape[2], x.shape[3]
        pad_F = (8 - (orig_F % 8)) % 8
        pad_T = (8 - (orig_T % 8)) % 8
        x = F.pad(x, (0, pad_T, 0, pad_F))

        # Forward Pass
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))

        bottleneck = self.aspp(self.pool3(e3))

        d1 = self.dec1(torch.cat([self.up1(bottleneck), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d1),         e2], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d2),         e1], dim=1))

        mask   = self.out_mask(d3)
        mask   = mask[:, :, :orig_F, :orig_T]
        x_orig = x[:, :, :orig_F, :orig_T]

        # Complex Multiplication
        x_real, x_imag = x_orig[:, 0], x_orig[:, 1]
        m_real, m_imag = mask[:, 0],   mask[:, 1]

        out_real    = (x_real * m_real) - (x_imag * m_imag)
        out_imag    = (x_real * m_imag) + (x_imag * m_real)
        out_complex = torch.complex(out_real, out_imag)

        # ISTFT Reconstruction
        return torch.istft(out_complex, n_fft=self.n_fft, hop_length=self.hop_length,
                           window=self.window, length=waveform.shape[1]).unsqueeze(1)

In [ ]:
# ==========================================
# 4. TRAINING LOOP & PLOT GENERATION
# ==========================================
def train_resunet_plus_plus():
    # 🚀 A100 SPEED UP: Enable CUDNN Benchmark and TF32
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"🚀 Launching Proposed ResUNet++ on: {device}")

    # --- SAVE DIRECTORIES ---
    BASE_DIR       = '/content/drive/MyDrive'
    CHECKPOINT_DIR = os.path.join(BASE_DIR, 'ResUNetPlusPlus_Results')
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

    # ⚠️ THIS IS THE EXACT PATH THAT WORKED FOR YOU EARLIER
    DATASET_ROOT = "/content/content/synthetic_dataset_v2"

    # 🚀 A100 SPEED UP: Increased batch_size to 16 and num_workers to 8
    train_loader, val_loader = get_dataloaders(DATASET_ROOT, batch_size=16, num_workers=8)

    model     = ResUNetPlusPlus().to(device)
    criterion = HybridLoss(weight_stft=0.4, weight_sdr=0.6).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
    epochs    = 50
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler    = GradScaler('cuda')

    best_val_sdr    = float('-inf')
    checkpoint_path = os.path.join(CHECKPOINT_DIR, 'best_resunet_plus_plus.pth')
    history_path    = os.path.join(CHECKPOINT_DIR, 'resunet_history.json')
    history         = {'train_loss': [], 'val_loss': [], 'train_sdr': [], 'val_sdr': []}

    # Early stopping parameters
    patience = 10
    no_improve_epochs = 0

    for epoch in range(1, epochs + 1):
        # --- Train ---
        model.train()
        train_loss, train_sdr = 0.0, 0.0
        loop = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs} [Train]", leave=False)
        for mixed, target in loop:
            mixed, target = mixed.to(device), target.to(device)
            optimizer.zero_grad()
            with autocast('cuda'):
                preds = model(mixed)
                loss, loss_sdr, _ = criterion(preds, target)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()
            train_sdr  += -loss_sdr.item()
            loop.set_postfix(loss=f"{loss.item():.4f}", sdr=f"{-loss_sdr.item():.3f}")

        avg_train_loss = train_loss / len(train_loader)
        avg_train_sdr  = train_sdr  / len(train_loader)

        # --- Validate ---
        model.eval()
        val_loss, val_sdr = 0.0, 0.0
        with torch.no_grad():
            for mixed, target in tqdm(val_loader, desc=f"Epoch {epoch}/{epochs} [Val]", leave=False):
                mixed, target = mixed.to(device), target.to(device)
                with autocast('cuda'):
                    preds = model(mixed)
                    loss, loss_sdr, _ = criterion(preds, target)
                val_loss += loss.item()
                val_sdr  += -loss_sdr.item()

        avg_val_loss = val_loss / len(val_loader)
        avg_val_sdr  = val_sdr  / len(val_loader)

        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['train_sdr'].append(avg_train_sdr)
        history['val_sdr'].append(avg_val_sdr)

        # Save history JSON continuously
        with open(history_path, 'w') as f: json.dump(history, f)
        scheduler.step()

        print(f"Epoch [{epoch}/{epochs}] | Train SDR: {avg_train_sdr:.3f} dB | Val SDR: {avg_val_sdr:.3f} dB")

        # Save Best Model
        if avg_val_sdr > best_val_sdr:
            best_val_sdr = avg_val_sdr
            torch.save(model.state_dict(), checkpoint_path)
            print(f"   🌟 Best ResUNet++ saved | Val SI-SDR: {best_val_sdr:.3f} dB")
            no_improve_epochs = 0
        else:
            no_improve_epochs += 1
            print(f"   ⚠️ No improvement for {no_improve_epochs} epoch(s).")
            if no_improve_epochs >= patience:
                print(f"🛑 Early stopping triggered! No improvement for {patience} epochs.")
                break

    # ==========================================
    # 🎉 TRAINING COMPLETE - GENERATE PLOTS
    # ==========================================
    print("Training Complete! Generating plots...")

    # 1. Loss Plot
    plt.figure(figsize=(10, 5))
    plt.plot(history['train_loss'], label='Train Loss', color='blue')
    plt.plot(history['val_loss'], label='Val Loss', color='orange')
    plt.title('Proposed ResUNet++: Hybrid Loss Over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(CHECKPOINT_DIR, 'loss_plot.png'))
    plt.close()

    # 2. SI-SDR Plot
    plt.figure(figsize=(10, 5))
    plt.plot(history['train_sdr'], label='Train SI-SDR', color='green')
    plt.plot(history['val_sdr'], label='Val SI-SDR', color='red')
    plt.title('Proposed ResUNet++: SI-SDR Improvement Over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('SI-SDR (dB)')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(CHECKPOINT_DIR, 'sdr_plot.png'))
    plt.close()

    print(f"✅ All files (Model, JSON History, and Plots) securely saved to {CHECKPOINT_DIR}")

# 🚀 RUN IT
train_resunet_plus_plus()

🚀 Launching Proposed ResUNet++ on: cuda


Epoch [1/50] | Train SDR: 0.236 dB | Val SDR: 1.810 dB
   🌟 Best ResUNet++ saved | Val SI-SDR: 1.810 dB


Epoch [2/50] | Train SDR: 2.170 dB | Val SDR: 1.969 dB
   🌟 Best ResUNet++ saved | Val SI-SDR: 1.969 dB


Epoch [3/50] | Train SDR: 2.700 dB | Val SDR: 2.196 dB
   🌟 Best ResUNet++ saved | Val SI-SDR: 2.196 dB


Epoch [4/50] | Train SDR: 3.148 dB | Val SDR: 2.650 dB
   🌟 Best ResUNet++ saved | Val SI-SDR: 2.650 dB


...

Epoch [43/50] | Train SDR: 7.382 dB | Val SDR: 4.106 dB
   ⚠️ No improvement for 6 epoch(s).


Epoch [44/50] | Train SDR: 7.401 dB | Val SDR: 4.104 dB
   ⚠️ No improvement for 7 epoch(s).


Epoch [45/50] | Train SDR: 7.416 dB | Val SDR: 4.127 dB
   ⚠️ No improvement for 8 epoch(s).


Epoch [46/50] | Train SDR: 7.423 dB | Val SDR: 4.128 dB
   ⚠️ No improvement for 9 epoch(s).


Epoch [47/50] | Train SDR: 7.422 dB | Val SDR: 4.129 dB
   ⚠️ No improvement for 10 epoch(s).
🛑 Early stopping triggered! No improvement for 10 epochs.
Training Complete! Generating plots...
✅ All files (Model, JSON History, and Plots) securely saved to /content/drive/MyDrive/ResUNetPlusPlus_Results


In [ ]:
import shutil
import os
from google.colab import files

# 1. Define the local path where your results are currently stuck
local_path = '/content/drive/MyDrive'
output_filename = 'All_Model_Results'

# 2. Zip everything inside that folder
print("Zipping all result folders... this might take a minute.")
shutil.make_archive(output_filename, 'zip', local_path)

# 3. Trigger the download to your computer
print("Zip complete. Starting download...")
files.download(f'{output_filename}.zip')

Zipping all result folders... this might take a minute.
Zip complete. Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>